In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [8]:
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")

real["label"] = 1
fake["label"] = 0

data = pd.concat([real, fake], ignore_index=True)

X = data["text"].fillna("")
y = data["label"]

data.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [10]:
lr = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("model", LogisticRegression())
])


In [11]:
nb = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("model", MultinomialNB())
])


In [12]:
svm = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("model", LinearSVC())
])

In [13]:
# create ensemble model
rf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

voting = VotingClassifier(
    estimators=[
        ("lr", lr),
        ("nb", nb),
        ("svm", svm)
    ],
    voting="hard"
)

In [14]:
# Train all models
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)
svm.fit(X_train, y_train)

rf.fit(X_train, y_train)
voting.fit(X_train, y_train)

VotingClassifier(estimators=[('lr',
                              Pipeline(steps=[('tfidf',
                                               TfidfVectorizer(max_features=5000)),
                                              ('model',
                                               LogisticRegression())])),
                             ('nb',
                              Pipeline(steps=[('tfidf',
                                               TfidfVectorizer(max_features=5000)),
                                              ('model', MultinomialNB())])),
                             ('svm',
                              Pipeline(steps=[('tfidf',
                                               TfidfVectorizer(max_features=5000)),
                                              ('model', LinearSVC())]))])

In [15]:
# compare model performance
lr_pred = lr.predict(X_test)
nb_pred = nb.predict(X_test)
svm_pred = svm.predict(X_test)

rf_pred = rf.predict(X_test)
voting_pred = voting.predict(X_test)

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "SVM",
        "Random Forest",
        "Voting Classifier"
    ],

    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, nb_pred),
        accuracy_score(y_test, svm_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, voting_pred)
    ],

    "Precision": [
        precision_score(y_test, lr_pred),
        precision_score(y_test, nb_pred),
        precision_score(y_test, svm_pred),
        precision_score(y_test, rf_pred),
        precision_score(y_test, voting_pred)
    ],

    "Recall": [
        recall_score(y_test, lr_pred),
        recall_score(y_test, nb_pred),
        recall_score(y_test, svm_pred),
        recall_score(y_test, rf_pred),
        recall_score(y_test, voting_pred)
    ],

    "F1 Score": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, nb_pred),
        f1_score(y_test, svm_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, voting_pred)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.987082,0.983751,0.989262,0.986499
1,Naive Bayes,0.934744,0.930200,0.933240,0.931718
2,SVM,0.994878,0.995557,0.993697,0.994626
3,Random Forest,0.997884,0.998131,0.997432,0.997782
4,Voting Classifier,0.988641,0.986279,0.989963,0.988117


In [16]:
# evalate prediction robustness
lr_cv = cross_val_score(lr, X, y, cv=3, scoring="accuracy")
nb_cv = cross_val_score(nb, X, y, cv=3, scoring="accuracy")
svm_cv = cross_val_score(svm, X, y, cv=3, scoring="accuracy")

rf_cv = cross_val_score(rf, X, y, cv=3, scoring="accuracy")
voting_cv = cross_val_score(voting, X, y, cv=3, scoring="accuracy")

In [17]:
robustness = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "SVM",
        "Random Forest",
        "Voting Classifier"
    ],

    "Mean Accuracy": [
        lr_cv.mean(),
        nb_cv.mean(),
        svm_cv.mean(),
        rf_cv.mean(),
        voting_cv.mean()
    ],

    "Standard Deviation": [
        lr_cv.std(),
        nb_cv.std(),
        svm_cv.std(),
        rf_cv.std(),
        voting_cv.std()
    ]
})

robustness

,Model,Mean Accuracy,Standard Deviation
0,Logistic Regression,0.968751,0.006589
1,Naive Bayes,0.881197,0.015958
2,SVM,0.984988,0.004502
3,Random Forest,0.993541,0.001207
4,Voting Classifier,0.970667,0.006465
